In [ ]:
# Cell 163 (07_rq3_grid.ipynb, cell 1): FRESH KERNEL, only this cell. Guards + validate_0pct + Part A thresholds at r=10 + mask precheck. Writes 1 file (exclusive create).
def _c163():
    import os, sys, json, time, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    from sklearn.base import clone
    from sklearn.model_selection import StratifiedKFold
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    OUT = "selection/tables/sel_rq3_thresholds_r10.json"; ss.check_path(os.path.join(R, OUT))
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    for rel, h in {"shared/src/masking.py": "c6094c2b232e", "shared/src/rq3.py": "5647309a230d", "shared/src/safe_save.py": "afff88e2b30a", "shared/src/checks.py": "ef542907c037"}.items():
        assert sha(R + "/" + rel).startswith(h), "hash mismatch for %s: STOP" % rel
    print("guards: PYTHONHASHSEED=0 and the four module hashes match")
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    ck = load("checks", "shared/src/checks.py"); rq3 = load("rq3", "shared/src/rq3.py")
    EXP = {"selection": dict(BN=0.6627, LR=0.6904, RF=0.6926, n_bn=30000), "monitoring": dict(BN=0.8519, LR=0.8618, RF=0.8603, n_bn=29998)}
    ctx = {}
    for c in ("selection", "monitoring"):
        print("\n=== validate_0pct: %s ===" % c); ctx[c] = rq3.load_context(c); rq3.validate_0pct(ctx[c], EXP[c])
    print("\nvalidation: ALL PASS")

    # Part A: thresholds at r = 10 on OUT-OF-FOLD TRAIN predictions
    cs = ctx["selection"]; Xtr = cs["base"]["Xtr"].values; y_tr = np.asarray(cs["g"]["y_train"]).astype(int); y_te = cs["y"]
    lr_gs = pickle_load = None
    import pickle
    lr_gs = pickle.load(open(R + "/selection/models/sel_lr_grid.pkl", "rb")); rf_gs = pickle.load(open(R + "/selection/models/sel_rf_grid.pkl", "rb"))
    print("\nCV-winner params: LR %s | RF %s" % (lr_gs.best_params_, rf_gs.best_params_))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = {"LR": np.zeros(len(y_tr)), "RF": np.zeros(len(y_tr))}; t0 = time.time()
    for k, (a, b) in enumerate(skf.split(Xtr, y_tr)):
        m = ck.fit_converged(clone(lr_gs.best_estimator_), Xtr[a], y_tr[a], name="LR fold %d" % k); oof["LR"][b] = m.predict_proba(Xtr[b])[:, 1]
        m = clone(rf_gs.best_estimator_).fit(Xtr[a], y_tr[a]); oof["RF"][b] = m.predict_proba(Xtr[b])[:, 1]
        print("  fold %d done (%.0fs)" % (k, time.time() - t0))
    grid = np.round(np.arange(1, 100) / 100.0, 2); R_ = 10
    cost = lambda p, y, t: R_ * int(((p < t) & (y == 1)).sum()) + int(((p >= t) & (y == 0)).sum())
    taus, rep = {"BN": 1.0 / 11.0}, {}
    for k in ("LR", "RF"):
        cc = np.array([cost(oof[k], y_tr, t) for t in grid]); best = grid[cc == cc.min()]; taus[k] = float(best[0])
        rep[k] = dict(tau=taus[k], plateau=[float(best.min()), float(best.max())], oof_cost=int(cc.min()), oof_accept_all=int(R_ * y_tr.sum()), oof_reject_all=int((y_tr == 0).sum()))
    print("\n=== Part A: selection thresholds at r = 10 (cost = 10*FN + FP, 0.01 grid, OOF train, test not used) ===")
    for k in ("LR", "RF"):
        r = rep[k]; fl = int((cs["p0"][k] >= r["tau"]).sum()); r["baseline_test_flagged"] = fl; r["baseline_test_flagged_share"] = fl / len(y_te)
        print("  %s: tau %.2f (plateau %.2f..%.2f) | OOF cost %d vs accept-all %d, reject-all %d | baseline test rows flagged: %d (%.2f%%)%s" % (
            k, r["tau"], *r["plateau"], r["oof_cost"], r["oof_accept_all"], r["oof_reject_all"], fl, 100 * fl / len(y_te), "   <- DEGENERATE: flips are 0 by construction" if fl == 0 else ""))
    fb = int((cs["p0"]["BN"] >= taus["BN"]).sum()); print("  BN: tau 1/11 = %.4f | baseline test rows flagged: %d (%.2f%%)" % (taus["BN"], fb, 100 * fb / len(y_te)))
    print("  (Phase 5 5:1 taus, reference only: LR %.2f, RF %.2f)" % (cs["tau"]["LR"], cs["tau"]["RF"]))

    # mask precheck: realised fractions of every mask the grid will draw
    MK = rq3.MK; LEV = (0.20, 0.40, 0.50); SEEDS = list(range(42, 52)); bad = []; mcar_dev = []
    for lev in LEV:
        for s in SEEDS:
            r = MK.make_mask(30000, lev, "mcar", s)[1]; mcar_dev.append(abs(r - lev))
            if abs(r - lev) > 0.005: bad.append(("mcar", lev, s, round(r, 4)))
    dep_fail = []
    for c in ("selection", "monitoring"):
        for v in rq3.MASK_VARS[c]:
            cc_ = rq3.cond_codes(ctx[c], v, "test")
            for lev in LEV:
                for s in SEEDS:
                    try: MK.make_mask(30000, lev, "dependent", s, cc_)
                    except AssertionError: dep_fail.append((c, v, lev, s))
    tr_dev = max(abs(MK.make_mask(119999, lev, "mcar", 42)[1] - lev) for lev in LEV)
    print("\n=== mask precheck (realised fraction vs level) ===")
    print("  MCAR test masks (3 levels x 10 seeds = 30 draws, shared by every variable): max |diff| %.4f | draws beyond 0.005: %d %s" % (max(mcar_dev), len(bad), bad))
    print("  dependent masks failing the 0.005 tolerance: %d of 180 %s" % (len(dep_fail), dep_fail[:5]))
    print("  MCAR train masks (seed 42, n=119,999) max |diff| %.4f" % tr_dev)
    print("  VERDICT:", "all masks within 0.005" if not bad and not dep_fail else "Part B assertion 'realised fraction within 0.005' WOULD FAIL for the MCAR draws listed: planning-instance decision needed before the grid")
    data = json.dumps(dict(r=10, rule="10*FN+FP, 0.01 grid, lowest tau at the minimum, 5-fold stratified OOF train (seed 42), CV-winner params, LR via fit_converged",
                           taus=taus, details=rep, bn_baseline_flagged=fb, phase5_reference_taus_5to1=dict(LR=cs["tau"]["LR"], RF=cs["tau"]["RF"]),
                           mask_precheck=dict(mcar_draws_beyond_0_005=bad, dependent_failures=dep_fail, mcar_max_dev=max(mcar_dev))), indent=1, default=float).encode()
    p = ss.check_path(os.path.join(R, OUT))
    with open(p, "xb") as f: f.write(data)
    assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
    print("\nsaved %s sha12 %s" % (OUT, hashlib.sha256(data).hexdigest()[:12]))
_c163()

In [ ]:
# Cell 164 (07_rq3_grid.ipynb): defines _grid() and runs the SELECTION grid (12 configs x 10 seeds). Resumable. Do not restart before Cell 165.
GRID_COLS = ["contribution", "variable", "level", "mechanism", "seed", "realised", "n_valid", "n_excluded", "n_patterns", "bp_maxdiff",
             "model", "variant", "imputation", "auc", "gini", "ks", "ap", "auc_0pct_same_rows", "delta_auc", "mean_abs_dp",
             "share_dp_gt_0_01", "flip_share", "tau", "run_seconds"]

def _grid(contribution):
    import os, sys, io, csv, json, time, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    for rel, h in {"shared/src/masking.py": "c6094c2b232e", "shared/src/rq3.py": "5647309a230d", "shared/src/safe_save.py": "afff88e2b30a", "shared/src/checks.py": "ef542907c037"}.items():
        assert sha(R + "/" + rel).startswith(h), "hash mismatch for %s: STOP" % rel
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    rq3 = load("rq3", "shared/src/rq3.py")
    c = contribution; assert c in ("selection", "monitoring")
    EXP = {"selection": dict(BN=0.6627, LR=0.6904, RF=0.6926, n_bn=30000), "monitoring": dict(BN=0.8519, LR=0.8618, RF=0.8603, n_bn=29998)}
    print("=== %s: guards OK; validate_0pct ===" % c)
    ctx = rq3.load_context(c); rq3.validate_0pct(ctx, EXP[c])
    if c == "selection":
        th = json.load(open(R + "/selection/tables/sel_rq3_thresholds_r10.json"))["taus"]
        assert abs(th["BN"] - 1 / 11) < 1e-12
        ctx["tau"] = {k: float(th[k]) for k in ("BN", "LR", "RF")}
    print("thresholds used:", ctx["tau"])
    PS = R + ("/selection/tables/sel_rq3_per_seed.csv" if c == "selection" else "/monitoring_rq3/mon_rq3_per_seed.csv")
    done = set()
    if os.path.exists(PS):
        ss.check_path(PS, overwrite=True)
        d0 = pd.read_csv(PS); d0["level"] = d0["level"].round(2)
        cnt = d0.groupby(["variable", "level", "mechanism", "seed"]).size()
        assert (cnt == 5).all(), "partial seed record in the per-seed file: STOP"
        done = set(cnt.index); print("resuming: %d seed records already on disk" % len(done))
        fh = open(PS, "a", newline="")
    else:
        ss.check_path(PS); fh = open(PS, "x", newline="")
        csv.DictWriter(fh, fieldnames=GRID_COLS).writeheader(); fh.flush(); os.fsync(fh.fileno())
    LEVELS, SEEDS, MECHS = (0.20, 0.40, 0.50), list(range(42, 52)), ("mcar", "dependent")
    TOL = {"mcar": 0.01, "dependent": 0.005}
    t_all = time.time(); n_new = 0; bp_max = 0.0
    for var in rq3.MASK_VARS[c]:
        for mech in MECHS:
            for lev in LEVELS:
                t_c = time.time(); devs = []; exs = []; ran = 0
                for seed in SEEDS:
                    if (var, round(lev, 2), mech, seed) in done: continue
                    t0 = time.time()
                    r = rq3.run_config(ctx, var, lev, mech, seed, secondary="median", bp_check=(seed == 42))
                    sec = time.time() - t0
                    assert abs(r["realised"] - lev) <= TOL[mech], "STOP: realised %.4f vs level %.2f (%s) exceeds %.3f" % (r["realised"], lev, mech, TOL[mech])
                    assert r["excluded"]["total"] / 30000 <= 0.001, "STOP: excluded rows > 0.1%"
                    if seed == 42:
                        assert r["bp_maxdiff"] is not None and r["bp_maxdiff"] <= 1e-10, "STOP: BP max diff %s" % r["bp_maxdiff"]
                        bp_max = max(bp_max, r["bp_maxdiff"])
                    recs = []
                    for x in r["rows"]:
                        assert all(np.isfinite(x[k]) for k in ("auc", "delta_auc", "mean_abs_dp", "share_dp_gt_0_01", "flip_share")), "STOP: non-finite metric"
                        variant = "BN" if x["model"] == "BN" else x["model"] + ("_primary" if x["imputation"].startswith("refit") else "_median")
                        recs.append(dict(contribution=c, variable=var, level=lev, mechanism=mech, seed=seed, realised=r["realised"], n_valid=x["n_valid"],
                                         n_excluded=r["excluded"]["total"], n_patterns=r["n_patterns"], bp_maxdiff=r["bp_maxdiff"], model=x["model"], variant=variant,
                                         imputation=x["imputation"], auc=x["auc"], gini=x["gini"], ks=x["ks"], ap=x["ap"], auc_0pct_same_rows=x["auc_0pct_same_rows"],
                                         delta_auc=x["delta_auc"], mean_abs_dp=x["mean_abs_dp"], share_dp_gt_0_01=x["share_dp_gt_0_01"], flip_share=x["flip_share"],
                                         tau=x["tau"], run_seconds=sec))
                    assert len(recs) == 5, "expected 5 records per seed run"
                    buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=GRID_COLS)
                    for rec in recs: w.writerow(rec)
                    fh.write(buf.getvalue()); fh.flush(); os.fsync(fh.fileno())
                    devs.append(abs(r["realised"] - lev)); exs.append(r["excluded"]["total"]); ran += 1; n_new += 1
                print("  %-26s %-9s level %.2f | %2d seeds run | max realised dev %.4f | max excluded %s | %.0fs" % (
                    var, mech, lev, ran, max(devs) if devs else 0.0, max(exs) if exs else "-", time.time() - t_c))
    fh.close()
    print("\n%s grid complete: %d new seed runs this session, %.1f min | BP max diff this session %.2e | per-seed file: %s" % (c, n_new, (time.time() - t_all) / 60, bp_max, os.path.relpath(PS, R)))

_grid("selection")

In [ ]:
# Cell 164d (07_rq3_grid.ipynb): diagnose IndexError in bin_natural_category. READ-ONLY, writes NOTHING. No restart needed.
def _c164d():
    import os, sys, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    rq3 = load("rq3", "shared/src/rq3.py")
    PS = R + "/selection/tables/sel_rq3_per_seed.csv"
    if os.path.exists(PS):
        d = pd.read_csv(PS); print("per-seed file: %d rows (%d complete seed runs); last row: %s" % (len(d), len(d) // 5, d.iloc[-1][["variable", "level", "mechanism", "seed"]].to_dict() if len(d) else "none"))
    else:
        print("per-seed file does not exist")
    NAT = ["30-59 Days Past Due", "90+ Days Late", "60-89 Days Past Due", "Number of Dependents", "Real Estate Loans"]
    for c in ("selection", "monitoring"):
        ctx = rq3.load_context(c); imp_cols = rq3._imp_cols(ctx); nat = [x for x in NAT if x in imp_cols]
        te0 = ctx["dfc"].loc[ctx["test_idx"], imp_cols].astype(float)
        if "Monthly Income" in imp_cols: te0["Monthly Income"] = np.log1p(te0["Monthly Income"])
        base_i = pd.DataFrame(ctx["base"]["imputer"].transform(te0) if ctx["base"]["imputer"] is not None else te0.values, columns=imp_cols)
        print("\n=== %s | natural count columns %s ===" % (c, nat))
        print("  0%% masking (locked imputer): min per column %s" % {x: round(float(base_i[x].min()), 2) for x in nat})
        for var in rq3.MASK_VARS[c]:
            for mech in ("mcar", "dependent"):
                for lev in (0.20, 0.40, 0.50):
                    cc = None if mech == "mcar" else rq3.cond_codes(ctx, var, "test")
                    mask, _ = rq3.MK.make_mask(len(te0), lev, mech, 42, cc)
                    imp, _, _, _ = rq3.get_refit_imputer(ctx, var, lev, mech)
                    te = te0.copy(); te.loc[mask, var] = np.nan
                    ti = pd.DataFrame(imp.transform(te), columns=imp_cols, index=te.index)
                    bad = {x: int((ti[x] <= -5).sum()) for x in nat if (ti[x] <= -5).any()}
                    mn = {x: round(float(ti[x].min()), 1) for x in nat}
                    med = te0.copy(); med.loc[mask, var] = rq3.__dict__.get("np", np).nan
                    flag = "  <-- CRASH CONDITION (value <= -5): %s" % bad if bad else ""
                    print("  %-24s %-9s %.2f | min imputed %s | masked-var range [%.3g, %.3g]%s" % (var, mech, lev, mn, float(ti.loc[mask, var].min()), float(ti.loc[mask, var].max()), flag))
_c164d()

In [ ]:
# Cell 167 (07_rq3_grid.ipynb): FRESH KERNEL, only this cell. Safe_save suites + rq3.py v2 (bin_mode, dependents logging, clip function) + validation. Saves only if everything passes.
CLIP_FUNC = '''


def run_clip_lr_rf(ctx, masked_raw, level, mechanism, seed):
    """SENSITIVITY ONLY: LR/RF with the primary refit imputer, imputed counts clipped at 0 before binning (bin_mode='clip').
    The BN is not rerun: the valid-row set is the baseline's (caller asserts the saved seed record shows no new BN exclusions).
    delta AUC is measured against the same saved 0% (wrap) predictions as the main grid."""
    y = ctx["y"]; n = len(y); p0 = ctx["p0"]
    cc = None if mechanism == "mcar" else cond_codes(ctx, masked_raw, "test")
    mask, realised = MK.make_mask(n, level, mechanism, seed, cc)
    valid = np.isfinite(p0["BN"])
    imp, _, _, _ = get_refit_imputer(ctx, masked_raw, level, mechanism)
    st = {}
    X = masked_matrix(ctx, masked_raw, mask, "refit", imp=imp, bin_mode="clip", stats=st)
    out = {}
    for k in ("LR", "RF"):
        p = ctx["models"][k].predict_proba(_xin(ctx["models"][k], X))[:, 1]
        out[k] = _row(ctx, k, "refit IterativeImputer (primary), clip", p, valid)
    return dict(realised=realised, rows=out, n_valid=int(valid.sum()))
'''

PATCHES = [
 ('''_L = lambda p: pickle.load(open(p, "rb"))''',
  '''_L = lambda p: pickle.load(open(p, "rb"))
RQ3_VERSION = "v2: bin_mode wrap|clip; Number of Dependents imputation logging; run_clip_lr_rf"'''),
 ('''def masked_matrix(ctx, masked_raw, test_mask, mode, imp=None):''',
  '''def _bin_natural(series, bin_mode="wrap", max_category=3):
    """Locked bin_natural_category, crash-free. 'wrap' (primary): labels[int(x)]; Python negative indexing for int(x) in -4..-1
    (identical to the locked function wherever that did not raise), int(x) % 4 for int(x) <= -5. 'clip' (sensitivity only): x clipped at 0 first."""
    if bin_mode not in ("wrap", "clip"):
        raise ValueError(bin_mode)
    labels = [str(i) for i in range(max_category)] + [f"{max_category}+"]
    def map_val(x):
        if pd.isnull(x):
            return np.nan
        if bin_mode == "clip":
            x = max(x, 0.0)
        if x < max_category:
            i = int(x)
            if i < -len(labels):
                i = i % len(labels)
            return labels[i]
        return labels[-1]
    return pd.Categorical(series.map(map_val), categories=labels, ordered=True)


def masked_matrix(ctx, masked_raw, test_mask, mode, imp=None, bin_mode="wrap", stats=None):'''),
 ('''    s_te = pd.DataFrame(index=te_i.index)''',
  '''    if stats is not None and "Number of Dependents" in te_i.columns:
        dv = te_i["Number of Dependents"].to_numpy()
        stats.update(test_dep_le_m1=int((dv <= -1).sum()), test_dep_le_m5=int((dv <= -5).sum()), test_dep_min=float(dv.min()))
    s_te = pd.DataFrame(index=te_i.index)'''),
 ('''g["bin_natural_category"](te_i[c], 3)''', '''_bin_natural(te_i[c], bin_mode, 3)'''),
 ('''ctx["imp_cache"][key] = (imp, t, real)''',
  '''if "Number of Dependents" in imp_cols:
        dv = imp.transform(tr)[:, imp_cols.index("Number of Dependents")]
        ctx.setdefault("train_stats", {})[key] = dict(train_dep_le_m1=int((dv <= -1).sum()), train_dep_le_m5=int((dv <= -5).sum()), train_dep_min=float(dv.min()))
    ctx["imp_cache"][key] = (imp, t, real)'''),
 ('''def run_config(ctx, masked_raw, level, mechanism, seed, secondary="median", bp_check=True, stop_frac=0.001, return_preds=False):''',
  '''def run_config(ctx, masked_raw, level, mechanism, seed, secondary="median", bp_check=True, stop_frac=0.001, return_preds=False, bin_mode="wrap"):'''),
 ('''t0 = time.time(); X = masked_matrix(ctx, masked_raw, mask, mode, imp=imp); T[lab + "_transform"] = time.time() - t0''',
  '''st = {}
        t0 = time.time(); X = masked_matrix(ctx, masked_raw, mask, mode, imp=imp, bin_mode=bin_mode, stats=st); T[lab + "_transform"] = time.time() - t0
        if lab == "primary":
            st.update(ctx.get("train_stats", {}).get((masked_raw, float(level), mechanism), {}))
        elif "Number of Dependents" in ctx["base"]["imp_tr"].columns:
            dv = ctx["base"]["imp_tr"]["Number of Dependents"].to_numpy()
            st.update(train_dep_le_m1=int((dv <= -1).sum()), train_dep_le_m5=int((dv <= -5).sum()), train_dep_min=float(dv.min()))'''),
 ('''rows.append(_row(ctx, k, "refit IterativeImputer (primary)" if lab == "primary" else "train %s (secondary)" % secondary, p, valid))''',
  '''r_ = _row(ctx, k, "refit IterativeImputer (primary)" if lab == "primary" else "train %s (secondary)" % secondary, p, valid); r_.update(st); rows.append(r_)'''),
]

def _c167():
    import os, sys, json, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    sys.path.insert(0, R + "/shared/src")
    for rel, h in {"shared/src/masking.py": "c6094c2b232e", "shared/src/safe_save.py": "afff88e2b30a", "shared/src/checks.py": "ef542907c037", "shared/src/rq3.py": "5647309a230d"}.items():
        assert sha(R + "/" + rel).startswith(h), "hash mismatch for %s: STOP" % rel
    import safe_save as ss
    P = R + "/shared/src/rq3.py"; BAK = R + "/shared/src/rq3_v1_5647309a230d.py.bak"; REC = "selection/tables/sel_rq3_module_v2.json"
    assert not os.path.exists(BAK), "backup exists: STOP"
    ss.check_path(os.path.join(R, REC))

    # 1. both safe_save refusal suites on the saved module
    S = ss.SaveRefused; c = ss.check_path
    def refused(fn):
        try: fn(); return False
        except S: return True
    def allowed(fn):
        try: fn(); return True
        except S: return False
    T = [("existing path refused", refused(lambda: c(R + "/selection/tables/sel_bn_metrics.json"))),
         ("full_variable/ refused", refused(lambda: c(R + "/full_variable/tables/sel_x.csv"))),
         ("archive/ refused", refused(lambda: c(R + "/archive/sel_x.csv"))),
         ("case-insensitive path refused", refused(lambda: c(R + "/Full_Variable/tables/sel_x.csv"))),
         ("missing prefix refused", refused(lambda: c(R + "/selection/tables/notprefixed.csv"))),
         ("sel_ in selection/ allowed", allowed(lambda: c(R + "/selection/tables/sel_zz_test.csv"))),
         ("sel_ in Data/processed allowed", allowed(lambda: c(R + "/Data/processed/sel_zz_test.pkl"))),
         ("mon_ in monitoring_rq3/ allowed", allowed(lambda: c(R + "/monitoring_rq3/mon_zz_test.csv"))),
         ("mon_ in selection/ refused", refused(lambda: c(R + "/selection/tables/mon_zz_test.csv"))),
         ("sel_ in monitoring_rq3/ refused", refused(lambda: c(R + "/monitoring_rq3/sel_zz_test.csv"))),
         ("mon_ in Data/processed refused", refused(lambda: c(R + "/Data/processed/mon_zz_test.pkl"))),
         ("mon_ in full_variable refused", refused(lambda: c(R + "/full_variable/mon_zz_test.csv"))),
         ("explicit prefix sel_ in monitoring_rq3 refused", refused(lambda: c(R + "/monitoring_rq3/sel_zz_test.csv", prefix="sel_"))),
         ("explicit prefix mon_ on a sel_ name refused", refused(lambda: c(R + "/selection/tables/sel_zz_test.csv", prefix="mon_"))),
         ("unknown prefix refused", refused(lambda: c(R + "/selection/tables/abc_zz_test.csv", prefix="abc_")))]
    for n, ok in T: print("  safe_save: %-50s %s" % (n, "PASS" if ok else "FAIL"))
    assert all(ok for _, ok in T), "safe_save suite failed: STOP"
    print("safe_save: 15/15 PASS\n")

    # 2. patch the saved module text in memory
    old = open(P).read(); new = old
    for a, b in PATCHES:
        assert new.count(a) == 1, "patch target not found exactly once: %s: STOP" % a[:70]
        new = new.replace(a, b)
    new = new + CLIP_FUNC
    ns = {}; exec(compile(new, "rq3_v2_candidate", "exec"), ns)
    print("candidate rq3 v2 built; sha12 %s (v1 %s)" % (hashlib.sha256(new.encode()).hexdigest()[:12], hashlib.sha256(old.encode()).hexdigest()[:12]))

    # 3. binning unit test against the locked function
    ctx = {k: ns["load_context"](k) for k in ("selection", "monitoring")}
    locked = ctx["selection"]["g"]["bin_natural_category"]
    vals = pd.Series([-7.3, -5.0, -4.99, -4.35, -1.84, -1.0, -0.5, 0.0, 0.7, 1.0, 2.9, 3.0, 8.2, np.nan])
    wr = pd.Series(ns["_bin_natural"](vals, "wrap", 3)).astype(str).tolist(); cl = pd.Series(ns["_bin_natural"](vals, "clip", 3)).astype(str).tolist()
    handled = (vals.isna() | (vals > -5)).to_numpy()
    lk = pd.Series(locked(vals[handled], 3)).astype(str).tolist()
    assert [w for w, h in zip(wr, handled) if h] == lk, "wrap differs from the locked function where the locked function works: STOP"
    lab = ["0", "1", "2", "3+"]
    for x, w in zip(vals, wr):
        if x <= -5: assert w == lab[int(x) % 4], "wrap rule violated"
    assert cl == pd.Series(locked(vals.clip(lower=0), 3)).astype(str).tolist(), "clip differs from locked(clipped values): STOP"
    print("binning test: wrap == locked on all %d handled values; <= -5 follows int(x) %% 4; clip == locked(clip(x, 0))" % int(handled.sum()))
    print("  value  wrap  clip: %s" % [(float(x), w, k) for x, w, k in zip(vals, wr, cl) if x <= -4 or pd.isna(x)])

    # 4. validate_0pct on the candidate, both contributions
    EXP = {"selection": dict(BN=0.6627, LR=0.6904, RF=0.6926, n_bn=30000), "monitoring": dict(BN=0.8519, LR=0.8618, RF=0.8603, n_bn=29998)}
    for k in ("selection", "monitoring"):
        print("\n=== validate_0pct (candidate v2): %s ===" % k); ns["validate_0pct"](ctx[k], EXP[k])

    # 5. save: backup, atomic replace, re-validate from disk, record
    with open(BAK, "xb") as f: f.write(old.encode())
    assert sha(BAK) == hashlib.sha256(old.encode()).hexdigest() and sha(BAK).startswith("5647309a230d")
    tmp = P + ".tmp_new"
    with open(tmp, "x", encoding="utf-8") as f: f.write(new)
    os.replace(tmp, P); os.chmod(P, 0o444)
    h2 = sha(P); assert h2 == hashlib.sha256(new.encode()).hexdigest()
    spec = importlib.util.spec_from_file_location("rq3_disk", P); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    for k in ("selection", "monitoring"):
        m.validate_0pct(m.load_context(k), EXP[k])
    print("\nsaved shared/src/rq3.py v2 sha12 %s (backup rq3_v1_5647309a230d.py.bak); re-validated from disk, both contributions" % h2[:12])
    data = json.dumps(dict(rq3_v2_sha12=h2[:12], rq3_v1_sha12="5647309a230d", rq3_v2_version=m.RQ3_VERSION, validation="validate_0pct passed for both contributions (candidate and from disk)",
                           safe_save_tests="15/15 PASS", wrap_rule="labels[int(x)] with Python negative indexing for -4..-1; int(x) % 4 for int(x) <= -5",
                           modules=dict(masking="c6094c2b232e", safe_save="afff88e2b30a", checks="ef542907c037")), indent=1).encode()
    p = ss.check_path(os.path.join(R, REC))
    with open(p, "xb") as f: f.write(data)
    print("saved %s sha12 %s" % (REC, hashlib.sha256(data).hexdigest()[:12]))
_c167()

In [ ]:
# Cell 168 (07_rq3_grid.ipynb): v2 driver + SELECTION grid (12 configs x 10 seeds), wrap mode. Resumable. Do not restart before Cell 169.
DEPCOLS = ["test_dep_le_m1", "test_dep_le_m5", "test_dep_min", "train_dep_le_m1", "train_dep_le_m5", "train_dep_min"]
GRID_COLS2 = ["contribution", "variable", "level", "mechanism", "seed", "realised", "n_valid", "n_excluded", "n_patterns", "bp_maxdiff",
              "model", "variant", "imputation", "auc", "gini", "ks", "ap", "auc_0pct_same_rows", "delta_auc", "mean_abs_dp",
              "share_dp_gt_0_01", "flip_share", "tau", "run_seconds"] + DEPCOLS
REG_COLS = ["realised", "n_valid", "n_excluded", "n_patterns", "bp_maxdiff", "auc", "gini", "ks", "ap", "auc_0pct_same_rows", "delta_auc", "mean_abs_dp", "share_dp_gt_0_01", "flip_share", "tau"]

def _grid2(contribution):
    import os, sys, io, csv, json, time, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    rec = json.load(open(R + "/selection/tables/sel_rq3_module_v2.json"))
    for rel, h in {"shared/src/masking.py": "c6094c2b232e", "shared/src/rq3.py": rec["rq3_v2_sha12"], "shared/src/safe_save.py": "afff88e2b30a", "shared/src/checks.py": "ef542907c037"}.items():
        assert sha(R + "/" + rel).startswith(h), "hash mismatch for %s: STOP" % rel
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    rq3 = load("rq3", "shared/src/rq3.py"); c = contribution; assert c in ("selection", "monitoring")
    EXP = {"selection": dict(BN=0.6627, LR=0.6904, RF=0.6926, n_bn=30000), "monitoring": dict(BN=0.8519, LR=0.8618, RF=0.8603, n_bn=29998)}
    print("=== %s: guards OK (rq3 v2 %s); validate_0pct ===" % (c, rec["rq3_v2_sha12"]))
    ctx = rq3.load_context(c); rq3.validate_0pct(ctx, EXP[c])
    if c == "selection":
        th = json.load(open(R + "/selection/tables/sel_rq3_thresholds_r10.json"))["taus"]
        assert abs(th["BN"] - 1 / 11) < 1e-12
        ctx["tau"] = {k: float(th[k]) for k in ("BN", "LR", "RF")}
    print("thresholds used:", ctx["tau"])
    PS = R + ("/selection/tables/sel_rq3_per_seed_v2.csv" if c == "selection" else "/monitoring_rq3/mon_rq3_per_seed_v2.csv")
    done = set()
    if os.path.exists(PS):
        ss.check_path(PS, overwrite=True)
        d0 = pd.read_csv(PS); d0["level"] = d0["level"].round(2)
        cnt = d0.groupby(["variable", "level", "mechanism", "seed"]).size()
        assert (cnt == 5).all(), "partial seed record in the per-seed file: STOP"
        done = set(cnt.index); print("resuming: %d seed records already on disk" % len(done)); fh = open(PS, "a", newline="")
    else:
        ss.check_path(PS); fh = open(PS, "x", newline="")
        csv.DictWriter(fh, fieldnames=GRID_COLS2).writeheader(); fh.flush(); os.fsync(fh.fileno())
    def reg_check():
        old = pd.read_csv(R + "/selection/tables/sel_rq3_per_seed.csv"); new = pd.read_csv(PS)
        f = lambda d: d[(d.variable == "Age") & (d.mechanism == "mcar") & (d.level.round(2) == 0.2) & (d.seed.isin([42, 43]))].set_index(["seed", "variant"]).sort_index()
        o, n2 = f(old), f(new)
        assert len(o) == 10 and len(n2) == 10, "regression rows missing: STOP"
        worst = max(float((o[k].astype(float).fillna(-999) - n2.loc[o.index, k].astype(float).fillna(-999)).abs().max()) for k in REG_COLS)
        assert worst <= 1e-12, "STOP: regression check failed, max abs diff %.3e vs the old rows (rq3 v1)" % worst
        print("  REGRESSION CHECK: Age/mcar/0.20 seeds 42-43 reproduce the old (rq3 v1) rows: all %d metric columns, max abs diff %.2e -> PASS" % (len(REG_COLS), worst))
    LEVELS, SEEDS, MECHS = (0.20, 0.40, 0.50), list(range(42, 52)), ("mcar", "dependent"); TOL = {"mcar": 0.01, "dependent": 0.005}
    t_all = time.time(); n_new = 0; bp_max = 0.0
    for var in rq3.MASK_VARS[c]:
        for mech in MECHS:
            for lev in LEVELS:
                t_c = time.time(); devs = []; exs = []; ran = 0; dmin = []
                for seed in SEEDS:
                    if (var, round(lev, 2), mech, seed) in done: continue
                    t0 = time.time()
                    r = rq3.run_config(ctx, var, lev, mech, seed, secondary="median", bp_check=(seed == 42), bin_mode="wrap"); sec = time.time() - t0
                    assert abs(r["realised"] - lev) <= TOL[mech], "STOP: realised %.4f vs level %.2f (%s) exceeds %.3f" % (r["realised"], lev, mech, TOL[mech])
                    assert r["excluded"]["total"] / 30000 <= 0.001, "STOP: excluded rows > 0.1%"
                    if seed == 42:
                        assert r["bp_maxdiff"] is not None and r["bp_maxdiff"] <= 1e-10, "STOP: BP max diff %s" % r["bp_maxdiff"]
                        bp_max = max(bp_max, r["bp_maxdiff"])
                    recs = []
                    for x in r["rows"]:
                        assert all(np.isfinite(x[k]) for k in ("auc", "delta_auc", "mean_abs_dp", "share_dp_gt_0_01", "flip_share")), "STOP: non-finite metric"
                        if x["model"] != "BN": assert all(x.get(k) is not None for k in DEPCOLS), "STOP: dependents logging missing"
                        variant = "BN" if x["model"] == "BN" else x["model"] + ("_primary" if x["imputation"].startswith("refit") else "_median")
                        recs.append(dict(contribution=c, variable=var, level=lev, mechanism=mech, seed=seed, realised=r["realised"], n_valid=x["n_valid"], n_excluded=r["excluded"]["total"],
                                         n_patterns=r["n_patterns"], bp_maxdiff=r["bp_maxdiff"], model=x["model"], variant=variant, imputation=x["imputation"], auc=x["auc"], gini=x["gini"],
                                         ks=x["ks"], ap=x["ap"], auc_0pct_same_rows=x["auc_0pct_same_rows"], delta_auc=x["delta_auc"], mean_abs_dp=x["mean_abs_dp"],
                                         share_dp_gt_0_01=x["share_dp_gt_0_01"], flip_share=x["flip_share"], tau=x["tau"], run_seconds=sec, **{k: x.get(k) for k in DEPCOLS}))
                        if x["model"] == "LR" and variant == "LR_primary": dmin.append(x["test_dep_min"])
                    assert len(recs) == 5, "expected 5 records per seed run"
                    buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=GRID_COLS2)
                    for rr in recs: w.writerow(rr)
                    fh.write(buf.getvalue()); fh.flush(); os.fsync(fh.fileno())
                    devs.append(abs(r["realised"] - lev)); exs.append(r["excluded"]["total"]); ran += 1; n_new += 1
                print("  %-26s %-9s level %.2f | %2d seeds run | max realised dev %.4f | max excluded %s | min imputed Dependents (test, primary) %s | %.0fs" % (
                    var, mech, lev, ran, max(devs) if devs else 0.0, max(exs) if exs else "-", ("%.2f" % min(dmin)) if dmin else "-", time.time() - t_c))
                if c == "selection" and var == "Age" and mech == "mcar" and abs(lev - 0.2) < 1e-9:
                    fh.flush(); reg_check()
    fh.close()
    print("\n%s grid complete: %d new seed runs this session, %.1f min | BP max diff this session %.2e | file: %s" % (c, n_new, (time.time() - t_all) / 60, bp_max, os.path.relpath(PS, R)))

_grid2("selection")

In [ ]:
# Cell 169 (07_rq3_grid.ipynb): MONITORING grid (24 configs x 10 seeds), wrap mode. Resumable. Needs _grid2 from Cell 168 (do not restart).
assert "_grid2" in globals(), "re-run Cell 168 (it defines _grid2; finished seeds are skipped) and then this cell"
_grid2("monitoring")

In [ ]:
# Cell 170 (07_rq3_grid.ipynb): aggregation + identity check + Dependents summary + figures + *_v2 files. Writes only if the identity check passes for BOTH contributions.
def _c170():
    import os, sys, io, json, hashlib, platform, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    import sklearn, pgmpy
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    rec = json.load(open(R + "/selection/tables/sel_rq3_module_v2.json"))
    assert hashlib.sha256(open(R + "/shared/src/rq3.py", "rb").read()).hexdigest().startswith(rec["rq3_v2_sha12"]), "rq3.py hash mismatch: STOP"
    VARIANTS = ["BN", "LR_primary", "RF_primary", "LR_median", "RF_median"]
    COL = {"BN": "tab:green", "LR": "tab:blue", "RF": "tab:orange"}
    STATED = {"selection": {"Age": 0.02187, "Debt Ratio": 0.01098},
              "monitoring": {"Credit Utilisation Rate": 0.0234, "30-59 Days Past Due": 0.0226, "90+ Days Late": 0.0214, "60-89 Days Past Due": 0.0106}}
    out, allok = {}, True
    for c in ("selection", "monitoring"):
        pre, fol = ("sel", "selection/tables/") if c == "selection" else ("mon", "monitoring_rq3/")
        PS, SM, NT = [fol + "%s_rq3_%s" % (pre, n) for n in ("per_seed_v2.csv", "summary_v2.csv", "notes_v2.json")]
        FG = "selection/figures/sel_rq3_auc_degradation_v2.png" if c == "selection" else "monitoring_rq3/mon_rq3_auc_degradation_v2.png"
        for rel in (SM, NT, FG): ss.check_path(os.path.join(R, rel))
        df = pd.read_csv(R + "/" + PS); df["level"] = df["level"].round(2)
        nconf = 12 if c == "selection" else 24
        assert len(df) == nconf * 50, "%s per-seed file incomplete: %d rows, expected %d. STOP" % (c, len(df), nconf * 50)
        print("\n================ %s: %d per-seed rows ================" % (c.upper(), len(df)))
        keys = ["variable", "level", "mechanism"]; rows = []
        for (v, l, m), g in df.groupby(keys, sort=False):
            piv = g.pivot(index="seed", columns="variant", values="delta_auc")
            for vt in VARIANTS:
                x = piv[vt]; sub = g[g.variant == vt]
                rows.append(dict(variable=v, level=l, mechanism=m, variant=vt, n_seeds=len(x), mean_delta_auc=x.mean(), sd_delta_auc=x.std(ddof=1), min_delta_auc=x.min(), max_delta_auc=x.max(),
                                 mean_flip_share=sub.flip_share.mean(), mean_abs_dp=sub.mean_abs_dp.mean(), mean_share_dp_gt_0_01=sub.share_dp_gt_0_01.mean(), mean_auc=sub.auc.mean(), n_bn_degrades_less=np.nan))
            for o in VARIANTS[1:]:
                d = piv["BN"] - piv[o]
                rows.append(dict(variable=v, level=l, mechanism=m, variant="BN_minus_" + o, n_seeds=len(d), mean_delta_auc=d.mean(), sd_delta_auc=d.std(ddof=1), min_delta_auc=d.min(), max_delta_auc=d.max(),
                                 mean_flip_share=np.nan, mean_abs_dp=np.nan, mean_share_dp_gt_0_01=np.nan, mean_auc=np.nan, n_bn_degrades_less=int((d > 0).sum())))
        S = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 200)
        pv = S[S.variant.isin(VARIANTS)].pivot_table(index=["variable", "mechanism", "level"], columns="variant", values="mean_delta_auc", sort=False)[VARIANTS]
        print("\nMean delta AUC vs 0%% masking (10 seeds; negative = degradation):"); print(pv.to_string(float_format=lambda x: "%.4f" % x))
        fl = S[S.variant.isin(VARIANTS)].pivot_table(index=["variable", "mechanism", "level"], columns="variant", values="mean_flip_share", sort=False)[VARIANTS]
        print("\nMean decision-flip share:"); print(fl.to_string(float_format=lambda x: "%.4f" % x))
        P = S[S.variant.str.startswith("BN_minus_")].pivot_table(index=["variable", "mechanism", "level"], columns="variant", values=["mean_delta_auc", "n_bn_degrades_less"], sort=False)
        print("\nPaired BN minus LR/RF delta AUC (positive = BN degrades less) and seeds out of 10 where the BN degrades less:"); print(P.to_string(float_format=lambda x: "%.4f" % x))

        pr = pd.read_pickle(R + ("/selection/predictions/sel_rq2_per_row.pkl" if c == "selection" else "/full_variable/per_row/vo_blanket_original_per_row.pkl"))
        if c == "selection": pr = pr["per_row"]
        dm = {v: float(pr[pr.variable == v + " (bin)"].delta.dropna().mean()) for v in df.variable.unique()}
        print("\nVoI Part A mean retraction delta per variable: computed %s | stated %s" % ({k: round(x, 5) for k, x in dm.items()}, STATED[c]))
        bn = df[df.variant == "BN"].groupby(keys, sort=False).agg(mean_abs_dp=("mean_abs_dp", "mean"), realised=("realised", "mean")).reset_index()
        bn["expected"] = bn.level * bn.variable.map(dm); bn["ratio"] = bn.mean_abs_dp / bn.expected; bn["ratio_using_realised"] = bn.mean_abs_dp / (bn.realised * bn.variable.map(dm))
        print("\nIDENTITY CHECK: BN mean |dp| / (level x mean retraction delta):"); print(bn.to_string(index=False, float_format=lambda x: "%.5f" % x))
        mc = bn[bn.mechanism == "mcar"]; dpd = bn[bn.mechanism == "dependent"]
        ok = bool(((mc.ratio >= 0.9) & (mc.ratio <= 1.1)).all()); allok &= ok
        print("MCAR ratios within 10%%: %s (min %.4f, max %.4f) | dependent ratios (descriptive): min %.4f, max %.4f" % ("PASS" if ok else "FAIL", mc.ratio.min(), mc.ratio.max(), dpd.ratio.min(), dpd.ratio.max()))

        b = df[df.variant == "BN"]; base_ex = 0 if c == "selection" else 2
        ex = b.groupby(keys + ["seed"]).n_excluded.max()
        print("\nExcluded rows per seed-run: min %d, max %d (%.4f%% of 30,000); runs above the baseline count (%d): %d" % (ex.min(), ex.max(), 100 * ex.max() / 30000, base_ex, int((ex > base_ex).sum())))
        print("max excluded by variable:", b.groupby("variable").n_excluded.max().to_dict())
        bpv = b.bp_maxdiff.dropna(); print("BP max diffs: %d checks (first seed of each configuration), max %.2e" % (len(bpv), bpv.max()))
        assert len(bpv) == nconf and bpv.max() <= 1e-10
        rt = b.run_seconds.sum() / 60; print("Total seed-run time recorded: %.1f min" % rt)
        md = b[b.mechanism == "mcar"].groupby(["level", "seed"]).realised.first().reset_index(); md["dev"] = (md.realised - md.level).abs(); lst = md[md.dev > 0.005]
        print("MCAR draws with |realised - level| > 0.005: %s" % lst.round(4).to_dict("records"))
        dep_dev = (b[b.mechanism == "dependent"].realised - b[b.mechanism == "dependent"].level).abs().max(); print("dependent max |realised - level| %.5f" % dep_dev)

        L = df[df.variant == "LR_primary"]
        dep = L.groupby(keys, sort=False).agg(test_le_m1_max=("test_dep_le_m1", "max"), test_le_m5_max=("test_dep_le_m5", "max"), seed_runs_with_le_m5=("test_dep_le_m5", lambda s: int((s > 0).sum())),
                                              test_min=("test_dep_min", "min"), train_le_m1_max=("train_dep_le_m1", "max"), train_le_m5_max=("train_dep_le_m5", "max"), train_min=("train_dep_min", "min")).reset_index()
        print("\nNumber of Dependents imputation (primary refit imputer; counts of rows with imputed value <= -1 / <= -5; per configuration over 10 seeds):"); print(dep.to_string(index=False, float_format=lambda x: "%.2f" % x))

        vars_ = list(df.variable.unique()); nr, nc = (1, 2) if len(vars_) == 2 else (2, 2)
        fig, axes = plt.subplots(nr, nc, figsize=(6.2 * nc, 4.6 * nr), squeeze=False)
        for ax, v in zip(axes.ravel(), vars_):
            for mech, ls in (("mcar", "-"), ("dependent", "--")):
                for md_ in ("BN", "LR_primary", "RF_primary"):
                    q = S[(S.variable == v) & (S.mechanism == mech) & (S.variant == md_)].sort_values("level"); col = COL[md_.split("_")[0]]
                    ax.plot(q.level, q.mean_delta_auc, ls=ls, marker="o", ms=4, color=col)
                    ax.fill_between(q.level, q.mean_delta_auc - q.sd_delta_auc, q.mean_delta_auc + q.sd_delta_auc, color=col, alpha=0.15, lw=0)
            ax.axhline(0, color="gray", lw=0.6); ax.set_title(v); ax.set_xticks([0.2, 0.4, 0.5]); ax.set_xlabel("Masking level (share of test rows)"); ax.set_ylabel("Change in AUC vs 0% masking")
            ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        axes.ravel()[0].legend(handles=[Line2D([0], [0], color=COL["BN"], label="BN"), Line2D([0], [0], color=COL["LR"], label="LR"), Line2D([0], [0], color=COL["RF"], label="RF"),
                                        Line2D([0], [0], color="k", ls="-", label="MCAR"), Line2D([0], [0], color="k", ls="--", label="dependent")], frameon=False, fontsize=9, loc="lower left")
        fig.suptitle("%s: AUC degradation under test-set masking of one variable" % c.capitalize())
        cap = "Lines = mean over 10 mask seeds (42-51); band = mean +/- SD across seeds, which reflects mask randomness only (fixed test set and trained models), not sampling uncertainty. LR and RF use the refit IterativeImputer (primary). BN removes the masked variable from the evidence. Dependent masking depends on Debt Ratio bin (Age bin when Debt Ratio is masked)."
        fig.tight_layout(rect=[0, 0.1, 1, 0.96]); fig.text(0.01, 0.005, textwrap.fill(cap, 150), fontsize=8, va="bottom")
        bio = io.BytesIO(); fig.savefig(bio, format="png", dpi=300, bbox_inches="tight"); plt.close(fig)
        taus = df.groupby("variant").tau.first().to_dict()
        notes = dict(contribution=c, rq3_grid=dict(variables=vars_, levels=[0.2, 0.4, 0.5], mechanisms=["mcar", "dependent"], seeds="42-51 (10)", configurations=nconf, seed_runs=nconf * 10, bin_mode="wrap"),
                     tolerance_rule="realised fraction: MCAR within 0.01, dependent within 0.005 (tolerance lives in the notebook; masking.py unchanged)",
                     mcar_draws_beyond_0_005=lst.round(5).to_dict("records"), dependent_max_dev=float(dep_dev),
                     conditioning_rule="dependent: p_i = min(1, s*w_i/mean(w)), w_i = conditioning bin index + 1; conditioning variable = Debt Ratio bin, except Age bin when the masked variable is Debt Ratio",
                     masks="drawn once per (variable, level, mechanism, seed) and shared by BN, LR, RF; test set only; models trained on complete data",
                     imputation=dict(primary="IterativeImputer(random_state=42) refit on a TRAIN copy with the same mask rule (train mask seed 42)", secondary="train median"),
                     binning="rq3 v2 bin_mode 'wrap': labels[int(x)] with Python negative indexing for -4..-1 (identical to the locked function), int(x) % 4 for int(x) <= -5",
                     taus=taus, tau_note=("BN 1/11 (r=10); LR/RF 0.57 from OOF train at r=10 (sel_rq3_thresholds_r10.json); Phase 5 5:1 taus (0.90, 0.69) reference only" if c == "selection" else "locked taus 0.17 / 0.69 / 0.67"),
                     sd_statement="SD across seeds reflects mask randomness only (fixed test set and trained models), not sampling uncertainty",
                     identity_check=dict(rule="MCAR: BN mean |dp| = level x mean VoI Part A retraction delta; within 10%", mcar_pass=ok, mcar_ratio_range=[float(mc.ratio.min()), float(mc.ratio.max())],
                                         dependent_ratio_range=[float(dpd.ratio.min()), float(dpd.ratio.max())], expected_delta_means=dm, table=bn.to_dict("records")),
                     dependents_imputation_summary=dep.to_dict("records"),
                     excluded_rows=dict(max_per_seed_run=int(ex.max()), baseline=base_ex, by_variable={k: int(v) for k, v in b.groupby("variable").n_excluded.max().items()}),
                     bp_max_diff=float(bpv.max()), seed_run_minutes=float(rt),
                     superseded=("sel_rq3_per_seed.csv (2 seed runs from rq3 v1 sha12 5647309a230d); regression check on Age/mcar/0.20 seeds 42-43 reproduced it exactly" if c == "selection" else "none"),
                     modules=dict(masking="c6094c2b232e", rq3="9ccbba66073b", safe_save="afff88e2b30a", checks="ef542907c037"),
                     versions=dict(python=platform.python_version(), pgmpy=pgmpy.__version__, sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__, PYTHONHASHSEED="0"))
        out[c] = {SM: S.to_csv(index=False).encode(), NT: json.dumps(notes, indent=1, default=float).encode(), FG: bio.getvalue()}
    if not allok:
        print("\nSTOP: identity check failed for at least one contribution. NOTHING WRITTEN. Report the identity tables above."); return
    for c in out:
        for rel, data in out[c].items():
            p = ss.check_path(os.path.join(R, rel))
            with open(p, "xb") as f: f.write(data)
            assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
            print("saved %-56s sha12 %s" % (rel, hashlib.sha256(data).hexdigest()[:12]))
    print("\nPART D FILES DONE")
_c170()

In [ ]:
# Cell 171 (07_rq3_grid.ipynb): clip sensitivity for LR/RF (primary imputer), seeds 42-46, all configurations. No BN rerun. Run after Cell 170. No restart.
def _c171():
    import os, sys, json, time, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    rec = json.load(open(R + "/selection/tables/sel_rq3_module_v2.json"))
    for rel, h in {"shared/src/masking.py": "c6094c2b232e", "shared/src/rq3.py": rec["rq3_v2_sha12"], "shared/src/safe_save.py": "afff88e2b30a", "shared/src/checks.py": "ef542907c037"}.items():
        assert sha(R + "/" + rel).startswith(h), "hash mismatch for %s: STOP" % rel
    OUT = {"selection": ("selection/tables/sel_rq3_clip_sensitivity.csv", "selection/tables/sel_rq3_clip_sensitivity_per_seed.csv"),
           "monitoring": ("monitoring_rq3/mon_rq3_clip_sensitivity.csv", "monitoring_rq3/mon_rq3_clip_sensitivity_per_seed.csv")}
    for a, b in OUT.values():
        for rel in (a, b): ss.check_path(os.path.join(R, rel))
    s = importlib.util.spec_from_file_location("rq3", R + "/shared/src/rq3.py"); rq3 = importlib.util.module_from_spec(s); s.loader.exec_module(rq3)
    PSF = {"selection": "selection/tables/sel_rq3_per_seed_v2.csv", "monitoring": "monitoring_rq3/mon_rq3_per_seed_v2.csv"}
    LEVELS, MECHS, SEEDS = (0.20, 0.40, 0.50), ("mcar", "dependent"), list(range(42, 47)); res = {}
    pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 200)
    for c in ("selection", "monitoring"):
        print("\n=== %s ===" % c); t0 = time.time()
        ctx = rq3.load_context(c)
        if c == "selection":
            th = json.load(open(R + "/selection/tables/sel_rq3_thresholds_r10.json"))["taus"]; ctx["tau"] = {k: float(th[k]) for k in ("BN", "LR", "RF")}
        df = pd.read_csv(R + "/" + PSF[c]); df["level"] = df["level"].round(2); base_ex = 0 if c == "selection" else 2
        valid = np.isfinite(ctx["p0"]["BN"]); recs = []; n_regr = n_skip = 0
        for var in rq3.MASK_VARS[c]:
            for mech in MECHS:
                cc = None if mech == "mcar" else rq3.cond_codes(ctx, var, "test")
                for lev in LEVELS:
                    imp, _, _, _ = rq3.get_refit_imputer(ctx, var, lev, mech)
                    for seed in SEEDS:
                        mask, realised = rq3.MK.make_mask(30000, lev, mech, seed, cc)
                        sv = df[(df.variable == var) & (df.level == round(lev, 2)) & (df.mechanism == mech) & (df.seed == seed)]
                        assert len(sv) == 5 and abs(sv.realised.iloc[0] - realised) <= 1e-12, "mask does not reproduce the saved realised fraction: STOP"
                        P = {}
                        for bm in ("wrap", "clip"):
                            st = {}; X = rq3.masked_matrix(ctx, var, mask, "refit", imp=imp, bin_mode=bm, stats=st)
                            for k in ("LR", "RF"):
                                p = ctx["models"][k].predict_proba(rq3._xin(ctx["models"][k], X))[:, 1]
                                P[(bm, k)] = rq3._row(ctx, k, "refit IterativeImputer (primary)", p, valid)
                        bn_d = float(sv[sv.variant == "BN"].delta_auc.iloc[0]); nex = int(sv.n_excluded.iloc[0])
                        for k in ("LR", "RF"):
                            sd = float(sv[sv.variant == k + "_primary"].delta_auc.iloc[0])
                            if nex == base_ex:
                                assert abs(P[("wrap", k)]["delta_auc"] - sd) <= 1e-12, "STOP: wrap on the baseline rows differs from the saved main-grid delta (%s %s %.2f %s %d %s)" % (var, mech, lev, mech, seed, k); n_regr += 1
                            else: n_skip += 1
                            recs.append(dict(contribution=c, variable=var, level=lev, mechanism=mech, seed=seed, model=k, n_excluded_main=nex, delta_wrap_same_rows=P[("wrap", k)]["delta_auc"],
                                             delta_clip=P[("clip", k)]["delta_auc"], delta_wrap_saved=sd, bn_delta_saved=bn_d, auc_wrap=P[("wrap", k)]["auc"], auc_clip=P[("clip", k)]["auc"],
                                             flip_wrap=P[("wrap", k)]["flip_share"], flip_clip=P[("clip", k)]["flip_share"], mean_abs_dp_wrap=P[("wrap", k)]["mean_abs_dp"], mean_abs_dp_clip=P[("clip", k)]["mean_abs_dp"]))
        R_ = pd.DataFrame(recs); R_["abs_diff"] = (R_.delta_clip - R_.delta_wrap_same_rows).abs()
        print("  %d runs (%d LR/RF rows) in %.0fs | wrap-on-baseline-rows regression vs saved main grid: %d exact matches, %d rows skipped (main run had new BN exclusions)" % (len(R_) // 2, len(R_), time.time() - t0, n_regr, n_skip))
        keys = ["variable", "level", "mechanism", "model"]; rows = []
        for (v, l, m, k), g in R_.groupby(keys, sort=False):
            mw, mc_, mws, bnm = g.delta_wrap_same_rows.mean(), g.delta_clip.mean(), g.delta_wrap_saved.mean(), g.bn_delta_saved.mean()
            sgn_w, sgn_c = np.sign(bnm - mws), np.sign(bnm - mc_); chg = bool(sgn_w != sgn_c)
            rows.append(dict(variable=v, level=l, mechanism=m, model=k, n_seeds=len(g), mean_delta_wrap_same_rows=mw, mean_delta_clip=mc_, diff_of_means=mc_ - mw, abs_diff_of_means=abs(mc_ - mw),
                             max_abs_diff_per_seed=g.abs_diff.max(), mean_abs_diff_per_seed=g.abs_diff.mean(), mean_delta_wrap_saved=mws, bn_mean_delta_saved=bnm,
                             bn_minus_model_wrap=bnm - mws, bn_minus_model_clip=bnm - mc_, sign_change=chg, flag_diff_gt_0_002=bool(g.abs_diff.max() > 0.002),
                             flagged=bool(g.abs_diff.max() > 0.002 or chg), mean_flip_wrap=g.flip_wrap.mean(), mean_flip_clip=g.flip_clip.mean()))
        S = pd.DataFrame(rows); res[c] = (S, R_)
        print("\nwrap vs clip, mean delta AUC per configuration and model (seeds 42-46):")
        print(S[["variable", "level", "mechanism", "model", "mean_delta_wrap_same_rows", "mean_delta_clip", "abs_diff_of_means", "max_abs_diff_per_seed", "mean_abs_diff_per_seed", "bn_minus_model_wrap", "bn_minus_model_clip", "sign_change", "flagged"]].to_string(index=False, float_format=lambda x: "%.5f" % x))
        print("\n%s: max abs per-seed diff %.5f | max abs diff of means %.5f | mean abs per-seed diff %.5f | configurations x models flagged: %d of %d (diff > 0.002: %d; sign change: %d)" % (
            c, S.max_abs_diff_per_seed.max(), S.abs_diff_of_means.max(), S.mean_abs_diff_per_seed.mean(), int(S.flagged.sum()), len(S), int(S.flag_diff_gt_0_002.sum()), int(S.sign_change.sum())))
        if S.flagged.any(): print("FLAGGED:"); print(S[S.flagged][["variable", "level", "mechanism", "model", "abs_diff_of_means", "max_abs_diff_per_seed", "sign_change"]].to_string(index=False, float_format=lambda x: "%.5f" % x))
    for c in res:
        for rel, d in zip(OUT[c], res[c]):
            data = d.to_csv(index=False).encode(); p = ss.check_path(os.path.join(R, rel))
            with open(p, "xb") as f: f.write(data)
            assert sha(p) == hashlib.sha256(data).hexdigest()
            print("saved %-62s sha12 %s" % (rel, sha(p)[:12]))
    print("\nCLIP SENSITIVITY DONE")
_c171()

In [ ]:
# Cell 172 (07_rq3_grid.ipynb): Phase 6b manifest/git DRY RUN. Writes NOTHING. No restart.
def _c172():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    def git(*a):
        return subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw = open(mp, 'rb').read()
    recs = list(csv.DictReader(open(mp, newline='')))
    existing = {r['path'] for r in recs}
    print("DRY_RUN = True")
    print("manifest rows now:", len(recs), "(expected 189) | ends with newline:", raw.endswith(b'\n'))
    tag = git("rev-list", "-n", "1", "full-variable-final")
    print("tag points to:", tag[:12], "| OK:", tag.startswith(TAG12))
    print("git status before:", git("status", "--porcelain").splitlines())
    rows = []
    for top in ("selection", "shared", "monitoring_rq3"):
        for dp, dn, fn in os.walk(os.path.join(R, top)):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in sorted(fn):
                if f.endswith(".pyc") or f == ".DS_Store":
                    continue
                p = os.path.join(dp, f); rel = os.path.relpath(p, R)
                if rel not in existing:
                    rows.append((rel, os.path.getsize(p), sha(p)[:12], time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))))
    print("\nrows to append (%d):" % len(rows))
    for r in sorted(rows): print("  ", *r)
    print("expected manifest rows after (new paths only):", len(recs) + len(rows), "| +1 more for the modified rq3.py")
    rq = [r for r in recs if r['path'] == "shared/src/rq3.py"]
    print("\nrq3.py rows in the manifest: %d (hash %s) | current sha12 %s" % (len(rq), [r.get('sha256', '')[:12] for r in rq], sha(os.path.join(R, "shared/src/rq3.py"))[:12]))
    print("old per-seed file (superseded, not deleted): selection/tables/sel_rq3_per_seed.csv in manifest:", "selection/tables/sel_rq3_per_seed.csv" in existing)
    print("\nwould stage ONLY: MANIFEST.csv, selection/notebooks/07_rq3_grid.ipynb, shared/src/rq3.py (not 03_-06_ notebooks, Reminder.md, Writing_Edits.md)")
    print("DRY RUN ONLY: nothing written.")
_c172()

In [ ]:
# Cell 173 (07_rq3_grid.ipynb): REAL RUN (approved). Manifest append (18 rows) + NEW git commit. Tag never touched.
def _c173():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    NOTE = "added after Phase 6b, 2026-10-15"
    NB = "selection/notebooks/07_rq3_grid.ipynb"; RQ = "shared/src/rq3.py"
    F = [("monitoring_rq3/mon_rq3_auc_degradation_v2.png", "18be5629f82c", "monitoring_rq3", ""),
         ("monitoring_rq3/mon_rq3_clip_sensitivity.csv", "d5c91f0f8316", "monitoring_rq3", ""),
         ("monitoring_rq3/mon_rq3_clip_sensitivity_per_seed.csv", "886ed3e516cc", "monitoring_rq3", ""),
         ("monitoring_rq3/mon_rq3_notes_v2.json", "57e5e0e688af", "monitoring_rq3", ""),
         ("monitoring_rq3/mon_rq3_per_seed_v2.csv", "3c2f34c84e21", "monitoring_rq3", ""),
         ("monitoring_rq3/mon_rq3_summary_v2.csv", "61d83e437de0", "monitoring_rq3", ""),
         ("selection/figures/sel_rq3_auc_degradation_v2.png", "d5a07ad3622d", "selection", ""),
         (NB, None, "selection", ""),
         ("selection/tables/sel_rq3_clip_sensitivity.csv", "979e16df7b83", "selection", ""),
         ("selection/tables/sel_rq3_clip_sensitivity_per_seed.csv", "8c2a0a4ad268", "selection", ""),
         ("selection/tables/sel_rq3_module_v2.json", "318951d55e2b", "selection", ""),
         ("selection/tables/sel_rq3_notes_v2.json", "b043cb23e059", "selection", ""),
         ("selection/tables/sel_rq3_per_seed.csv", "4323d0aef92b", "selection", "; SUPERSEDED, not deleted: 2 seed runs (Age/mcar/0.20 seeds 42-43) from rq3 v1 5647309a230d before the IndexError stop; reproduced exactly by sel_rq3_per_seed_v2.csv"),
         ("selection/tables/sel_rq3_per_seed_v2.csv", "d3bd391629e6", "selection", ""),
         ("selection/tables/sel_rq3_summary_v2.csv", "61b13d2e91a6", "selection", ""),
         ("selection/tables/sel_rq3_thresholds_r10.json", "7a826300744f", "selection", ""),
         ("shared/src/rq3_v1_5647309a230d.py.bak", "5647309a230d", "shared", "; backup of rq3.py v1")]
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0:
            raise RuntimeError(f"git {' '.join(a)} failed: {r.stderr.strip()[:300]}")
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw0 = open(mp, 'rb').read()
    recs = list(csv.DictReader(open(mp, newline='')))
    existing = {r['path'] for r in recs}
    assert raw0.endswith(b'\n') and len(recs) == 189, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12), "tag moved: STOP"
    rq_new = sha(os.path.join(R, RQ))
    assert rq_new.startswith("9ccbba66073b"), "rq3.py is not the validated v2: STOP"
    assert not any(r['path'] == RQ and r.get('sha256', '').startswith("9ccbba66073b") for r in recs), "rq3 v2 row already present: STOP"
    rows = []
    for rel, h12, role, extra in F:
        p = os.path.join(R, rel)
        assert rel not in existing and os.path.exists(p), f"unexpected state for {rel}: STOP"
        h = sha(p)
        if h12 is not None:
            assert h.startswith(h12), f"hash changed since dry run for {rel}: STOP"
        else:
            print("notebook hash recorded now: %s (dry run showed e578f1bd754d)" % h[:12])
        rows.append([rel, os.path.getsize(p), h, role, time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), NOTE + extra])
    p = os.path.join(R, RQ)
    rows.append([RQ, os.path.getsize(p), rq_new, "shared", time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))),
                 NOTE + "; MODIFIED (v2): bin_mode wrap|clip, Number of Dependents logging, run_clip_lr_rf; supersedes the earlier rq3.py row (v1 kept as rq3_v1_5647309a230d.py.bak)"])
    with open(mp, 'a', newline='') as fh:
        csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=''))))
    assert n == 207 and open(mp, 'rb').read().startswith(raw0), "manifest check failed: STOP (not committed)"
    git("add", "MANIFEST.csv", NB, RQ)
    ident = []
    if not git("config", "user.name", check=False) or not git("config", "user.email", check=False):
        ident = ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 6b: RQ3 full grid (rq3 v2 wrap/clip), per-seed/summary/notes v2, clip sensitivity, figures")
    print("manifest rows now:", n, "(expected 207)")
    print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12))
    print("git status:", git("status", "--porcelain").splitlines())
_c173()